# A/B Testing

In [ ]:
# datascience
from datascience import *
import numpy as np

%matplotlib inline
import matplotlib.pyplot as plots
plots.style.use('fivethirtyeight')

import warnings
warnings.simplefilter("ignore")

In [ ]:
# polars
import polars as pl
from IPython.display import display
import numpy as np

def _append(base, extra):
    """Append a value or a Series onto a Series."""
    extra_s = extra if isinstance(extra, pl.Series) else pl.Series([extra])
    if base.len() == 0:
        return extra_s
    if base.dtype != extra_s.dtype:
        try:
            extra_s = extra_s.cast(base.dtype, strict=False)
        except Exception:
            base = base.cast(extra_s.dtype, strict=False)
    return pl.concat([base, extra_s])


## Comparing Two Samples

In [ ]:
# datascience
births = Table.read_table('baby.csv')

In [ ]:
# polars
births_pl = pl.read_csv('baby.csv', eol_char='\r')

In [ ]:
# datascience
births

In [ ]:
# polars
births_pl

In [ ]:
# datascience
smoking_and_birthweight = births.select('Maternal Smoker', 'Birth Weight')

In [ ]:
# polars
smoking_and_birthweight_pl = births_pl.select('Maternal Smoker', 'Birth Weight')

In [ ]:
# datascience
smoking_and_birthweight.group('Maternal Smoker')

In [ ]:
# polars
(lambda _df, _k='Maternal Smoker': _df.group_by(_k).len(name='count').sort(_k))(smoking_and_birthweight_pl)

In [ ]:
# datascience
smoking_and_birthweight.hist('Birth Weight', group='Maternal Smoker')

In [ ]:
# polars
# matplotlib
_df = smoking_and_birthweight_pl
for _key, _part in _df.group_by('Maternal Smoker'):
    _label = _key[0] if isinstance(_key, tuple) else _key
    plots.hist(_part.get_column('Birth Weight').to_numpy(), density=True, histtype='stepfilled', alpha=0.7, label=str(_label))
plots.xlabel('Birth Weight')
plots.ylabel("Percent per unit")
plots.gca().yaxis.set_major_formatter(lambda x, pos: "{:g}".format(100 * x))
plots.legend()

## Test Statistic

[Question] What values of our statistic are in favor of the alternative: positive or negative?

In [ ]:
# datascience
means_table = smoking_and_birthweight.group('Maternal Smoker', np.average)
means_table

In [ ]:
# polars
means_table_pl = (lambda _df, _k='Maternal Smoker': _df.group_by(_k).agg([pl.col(c).mean().alias(f"{c} average") for c in _df.columns if c != _k and _df[c].dtype.is_numeric()]).sort(_k))(smoking_and_birthweight_pl)
means_table_pl

In [ ]:
# datascience
means = means_table.column(1)
observed_difference = means.item(1) - means.item(0)
observed_difference

In [ ]:
# polars
means_pl = means_table_pl.to_series(1)
observed_difference_pl = means_pl.item(1) - means_pl.item(0)
observed_difference_pl

In [ ]:
# datascience
def difference_of_means(table, numeric_label, category_label):
    """
    Takes: 
       - name of table
       - column label of numerical variable
       - column label of categorical variable
       
    Returns: Difference of means of the two groups
    """
    
    #table with the two relevant columns
    reduced = table.select(numeric_label, category_label)  
    
    # table containing group means
    means_table = reduced.group(category_label, np.average)
    
    # array of group means
    means = means_table.column(1)
    
    return means.item(1) - means.item(0)

In [ ]:
# polars
def difference_of_means_pl(table, numeric_label, category_label):
    """
    Takes: 
       - name of table
       - column label of numerical variable
       - column label of categorical variable
       
    Returns: Difference of means of the two groups
    """
    
    #table with the two relevant columns
    reduced = table.select(numeric_label, category_label)  
    
    # table containing group means
    means_table_pl = (lambda _df, _k=category_label: _df.group_by(_k).agg([pl.col(c).mean().alias(f"{c} average") for c in _df.columns if c != _k and _df[c].dtype.is_numeric()]).sort(_k))(reduced)
    
    # array of group means
    means_pl = means_table_pl.to_series(1)
    
    return means_pl.item(1) - means_pl.item(0)

In [ ]:
# datascience
difference_of_means(births, 'Birth Weight', 'Maternal Smoker')

In [ ]:
# polars
difference_of_means_pl(births_pl, 'Birth Weight', 'Maternal Smoker')

## Random Permutation (Shuffling)

In [ ]:
# datascience
staff = Table().with_columns(
    'Names', make_array('Jim', 'Pam', 'Dwight', 'Michael'),
    'Ages', make_array(29, 28, 34, 41)
)

In [ ]:
# polars
staff_pl = pl.DataFrame({'Names': pl.Series("values", ['Jim', 'Pam', 'Dwight', 'Michael']), 'Ages': pl.Series("values", [29, 28, 34, 41])})

In [ ]:
# datascience
staff.sample()

In [ ]:
# polars
(lambda _df: _df.sample(n=_df.height, with_replacement=True, shuffle=True))(staff_pl)

In [ ]:
# datascience
staff.sample(with_replacement = False)

In [ ]:
# polars
(lambda _df: _df.sample(n=_df.height, with_replacement=False, shuffle=True))(staff_pl)

In [ ]:
# datascience
staff.with_column('Shuffled', staff.sample(with_replacement = False).column(0))

In [ ]:
# polars
staff_pl.with_columns(pl.Series('Shuffled', (lambda _df: _df.sample(n=_df.height, with_replacement=False, shuffle=True))(staff_pl).to_series(0)))

## Simulation Under Null Hypothesis

In [ ]:
# datascience
smoking_and_birthweight

In [ ]:
# polars
smoking_and_birthweight_pl

In [ ]:
# datascience
shuffled_labels = smoking_and_birthweight.sample(with_replacement=False).column('Maternal Smoker')

In [ ]:
# polars
shuffled_labels_pl = (lambda _df: _df.sample(n=_df.height, with_replacement=False, shuffle=True))(smoking_and_birthweight_pl).get_column('Maternal Smoker')

In [ ]:
# datascience
original_and_shuffled = smoking_and_birthweight.with_column(
    'Shuffled Label', shuffled_labels
)

In [ ]:
# polars
original_and_shuffled_pl = smoking_and_birthweight_pl.with_columns(pl.Series('Shuffled Label', shuffled_labels_pl))

In [ ]:
# datascience
original_and_shuffled

In [ ]:
# polars
original_and_shuffled_pl

In [ ]:
# datascience
difference_of_means(original_and_shuffled, 'Birth Weight', 'Shuffled Label')

In [ ]:
# polars
difference_of_means_pl(original_and_shuffled_pl, 'Birth Weight', 'Shuffled Label')

In [ ]:
# datascience
difference_of_means(original_and_shuffled, 'Birth Weight', 'Maternal Smoker')

In [ ]:
# polars
difference_of_means_pl(original_and_shuffled_pl, 'Birth Weight', 'Maternal Smoker')

## Permutation Test

In [ ]:
# datascience
def one_simulated_difference(table, numeric_label, category_label):
    """
    Takes: 
       - name of table
       - column label of numerical variable
       - column label of categorical variable
       
    Returns: Difference of means of the two groups
    """
    
    # array of shuffled labels
    shuffled_labels = table.sample(with_replacement = False).column(category_label)
    
    # table of numerical variable and shuffled labels
    shuffled_table = table.select(numeric_label).with_column('Shuffled Label', shuffled_labels)
    
    return difference_of_means(shuffled_table, numeric_label, 'Shuffled Label')   

In [ ]:
# polars
def one_simulated_difference_pl(table, numeric_label, category_label):
    """
    Takes: 
       - name of table
       - column label of numerical variable
       - column label of categorical variable
       
    Returns: Difference of means of the two groups
    """
    
    # array of shuffled labels
    shuffled_labels_pl = (lambda _df: _df.sample(n=_df.height, with_replacement=False, shuffle=True))(table).get_column(category_label)
    
    # table of numerical variable and shuffled labels
    shuffled_table = table.select(numeric_label).with_columns(pl.Series('Shuffled Label', shuffled_labels_pl))
    
    return difference_of_means_pl(shuffled_table, numeric_label, 'Shuffled Label')   

In [ ]:
# datascience
one_simulated_difference(births, 'Birth Weight', 'Maternal Smoker')

In [ ]:
# polars
one_simulated_difference_pl(births_pl, 'Birth Weight', 'Maternal Smoker')

In [ ]:
# datascience
differences = make_array()

for i in np.arange(2000):
    new_difference = one_simulated_difference(births, 'Birth Weight', 'Maternal Smoker')
    differences = np.append(differences, new_difference)

In [ ]:
# polars
differences_pl = []

for i in pl.int_range(2000, eager=True):
    new_difference = one_simulated_difference_pl(births_pl, 'Birth Weight', 'Maternal Smoker')
    differences_pl.append(new_difference)
differences_pl = pl.Series("values", differences_pl)

In [ ]:
# datascience
Table().with_column('Difference Between Group Means', differences).hist()
plots.ylim(-.02, .4)
plots.scatter(observed_difference, 0, color='red', s=30);
print('Observed Difference:', observed_difference)
plots.title('Prediction Under the Null Hypothesis');

In [ ]:
# polars
# matplotlib
_df = pl.DataFrame({'Difference Between Group Means': differences_pl})
_cols = [c for c in _df.columns if _df[c].dtype.is_numeric()]
if len(_cols) == 1:
    plots.hist(_df.get_column(_cols[0]).to_numpy(), density=True)
    plots.xlabel(_cols[0])
else:
    for _col in _cols:
        plots.hist(_df.get_column(_col).to_numpy(), density=True, label=_col)
    plots.legend()
plots.ylabel("Percent per unit")
plots.gca().yaxis.set_major_formatter(lambda x, pos: "{:g}".format(100 * x))
plots.ylim(-.02, .4)
plots.scatter(observed_difference_pl, 0, color='red', s=30);
print('Observed Difference:', observed_difference_pl)
plots.title('Prediction Under the Null Hypothesis');